In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages=messages,text=prompt)
    add_assistant_message(messages=messages, text="```json")
    text = chat(messages=messages, stop_sequences=["```"])
    return json.loads(text)

In [5]:
dataset = generate_dataset()
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [6]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt  = f"""
        please solve the following task:
        {test_case["task"]}
    """
    messages = []
    add_user_message(messages=messages,text=prompt)
    output = chat(messages=messages)
    return output


In [7]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case=test_case)
    score = 10
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }



In [8]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case=test_case)
        results.append(result)
    return results

In [9]:
with open("dataset.json", "r") as f :
    dataset = json.load(f)
results  = run_eval(dataset=dataset)


In [11]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS IAM ARN Validator\n\n```python\nimport re\n\ndef validate_iam_arn(arn: str) -> bool:\n    \"\"\"\n    Validates an AWS IAM ARN string.\n    \n    Expected format: arn:aws:iam::account-id:role/role-name\n    \n    Args:\n        arn: The ARN string to validate\n        \n    Returns:\n        True if the ARN is valid, False otherwise\n    \"\"\"\n    if not isinstance(arn, str):\n        return False\n    \n    # AWS IAM ARN pattern\n    # arn:aws:iam::account-id:role/role-name\n    pattern = r'^arn:aws:iam::\\d{12}:role/[a-zA-Z0-9\\-_./]+$'\n    \n    return bool(re.match(pattern, arn))\n\n\n# Test cases\nif __name__ == \"__main__\":\n    # Valid ARNs\n    valid_arns = [\n        \"arn:aws:iam::123456789012:role/MyRole\",\n        \"arn:aws:iam::999999999999:role/service-role/my-service-role\",\n        \"arn:aws:iam::000000000000:role/test-role-123\",\n        \"arn:aws:iam::123456789012:role/path/to/role-name\",\n    ]\n    \n    # Invalid ARNs\n    invalid